# fedOPD - federation under stress: partial participation and the price of privacy noise

`06`/`08` federate under two generous assumptions: every one of the 141 trusts participates in every
round, and client updates travel in the clear (FedAvg is data governance, not privacy - weights leak
information about the data that made them). This notebook measures what happens when both
assumptions are relaxed, on the `08` reference configuration (MLP, 15 rounds x 2 local epochs,
seed 0, validation window).

**Arms, declared before the run.** Participation 100% / 50% / 20% of clients sampled per round.
Privacy: each client update clipped to the median round-1 update norm, then server-side Gaussian
noise at multiplier sigma in {0, 0.5, 1.0} of the clipping bound over the participant count.
The clipping bound rule is declared here, not chosen after seeing results.

**What is and is not claimed.** Reported is the utility cost (MASE, up-call precision) of clipping
and noise. A formal (epsilon, delta) guarantee needs a privacy accountant over the full training
run and is **not** claimed; the sigma values are reported as what they are - noise multipliers.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

torch.manual_seed(0)
rng = np.random.default_rng(0)                         # client sampling, separate from torch's stream

CANON = Path("../data/canonical")
panel = pd.read_parquet(CANON / "rtt_panel.parquet")
index = pd.read_csv(CANON / "series_index.csv", parse_dates=["split_month"])

SPLIT = index.split_month.iloc[0]
H, L, HIDDEN, ROUNDS, LOCAL_EPOCHS, BAND = 12, 24, 128, 15, 2, 0.05
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

wide = panel.pivot(index=["provider_code", "dept"], columns="month", values="demand")
wide = wide.loc[list(zip(index.provider_code, index.dept))]
wide.columns = pd.to_datetime(wide.columns)
train_cols = wide.columns < SPLIT
score_cols = (wide.columns >= SPLIT) & (wide.columns < SPLIT + pd.DateOffset(months=H))
assert score_cols.sum() == H, "scored window is not H months"

hist = wide.loc[:, train_cols].to_numpy(dtype=np.float32)
future = wide.loc[:, score_cols].to_numpy(dtype=np.float32)
trusts = index.provider_code.to_numpy()
norm = hist.mean(axis=1) + 1.0
month_ix = np.array([m.month - 1 for m in wide.columns[train_cols]])
eye = np.eye(12, dtype=np.float32)
scale = index.mase_scale.to_numpy()

ref = hist[:, -12:].mean(axis=1)
truth_growth = future.mean(axis=1) / np.where(ref > 0, ref, np.nan) - 1
eligible = np.isfinite(truth_growth)
truth_up = truth_growth > BAND
base_rate = truth_up[eligible].mean() * 100
print(f"base rate for the up call: {base_rate:.1f}% of {int(eligible.sum()):,} services")
print(f"denominators: MASE over {int((scale > 0).sum()):,} | direction over {int(eligible.sum()):,} | {len(wide):,} series total")

base rate for the up call: 45.9% of 1,978 services
denominators: MASE over 2,070 | direction over 1,978 | 2,085 series total


In [2]:
def new_model():
    return nn.Sequential(nn.Linear(L + 12, HIDDEN), nn.ReLU(), nn.Dropout(0.1),
                         nn.Linear(HIDDEN, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, H)).to(device)

def get_w(m): return [p.detach().clone() for p in m.parameters()]
def set_w(m, w):
    with torch.no_grad():
        for p, q in zip(m.parameters(), w): p.copy_(q)

def train_local(model, X, Y, epochs, lr=1e-3):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()
    X, Y = X.to(device), Y.to(device)
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 256):
            b = perm[i:i + 256]
            opt.zero_grad(); loss_fn(model(X[b]), Y[b]).backward(); opt.step()

clients = {}
for code_ in np.unique(trusts):
    rows = np.flatnonzero(trusts == code_)
    xs, ys = [], []
    for r in rows:
        s = hist[r] / norm[r]
        for t in range(len(s) - L - H + 1):
            xs.append(np.concatenate([s[t:t + L], eye[month_ix[t + L - 1]]]))
            ys.append(s[t + L:t + L + H])
    clients[code_] = (torch.tensor(np.array(xs)), torch.tensor(np.array(ys)))
X_test = torch.tensor(np.concatenate(
    [hist[:, -L:] / norm[:, None], np.repeat(eye[month_ix[-1]][None, :], len(hist), axis=0)], axis=1))
codes = list(clients)
sizes = np.array([len(clients[c][0]) for c in codes], dtype=float)

def evaluate(model):
    model.eval()
    with torch.no_grad():
        pred = np.clip(model(X_test.to(device)).cpu().numpy() * norm[:, None], 0, None)
    mae = np.abs(pred - future).mean(axis=1)
    mase = np.nanmedian(np.where(scale > 0, mae / np.where(scale > 0, scale, 1), np.nan))
    g = pred.mean(axis=1) / np.where(ref > 0, ref, np.nan) - 1
    called = np.isfinite(g) & eligible & (g > BAND)
    prec = truth_up[called].mean() * 100 if called.sum() else np.nan
    return mase, prec, int(called.sum())

print(f"{len(codes)} clients ready")

141 clients ready


## 1. Partial participation

A round samples a fraction of clients uniformly at random; the average is over the sampled clients'
window counts. 15 rounds regardless of the fraction, so a 20% run sees each client about 3 times -
that is the realistic cost being measured, not a budget-matched abstraction.

In [3]:
def fedavg(frac=1.0, clip=False, sigma=0.0):
    torch.manual_seed(0)                               # each arm starts from the same init and stream
    w = get_w(new_model())
    model = new_model()
    m = max(1, int(round(frac * len(codes))))
    clip_c = None                                      # set by the round-1 probe when clipping is on
    for r in range(ROUNDS):
        picked = rng.choice(len(codes), size=m, replace=False) if frac < 1.0 else np.arange(len(codes))
        alpha = sizes[picked] / sizes[picked].sum()
        deltas = []
        for a, ci in zip(alpha, picked):
            set_w(model, w)
            train_local(model, clients[codes[ci]][0], clients[codes[ci]][1], LOCAL_EPOCHS)
            deltas.append(([p - q for p, q in zip(get_w(model), w)], a))
        if clip:
            norms = [torch.sqrt(sum((d ** 2).sum() for d in ds)).item() for ds, _ in deltas]
            if clip_c is None:
                clip_c = float(np.median(norms))       # the declared rule: median norm of round 1
            deltas = [([d * min(1.0, clip_c / max(n, 1e-12)) for d in ds], a)
                      for (ds, a), n in zip(deltas, norms)]
        acc = [torch.zeros_like(p) for p in w]
        for ds, a in deltas:
            for i, d in enumerate(ds):
                acc[i] += a * d
        if sigma > 0:
            for i in range(len(acc)):                  # server-side Gaussian noise on the mean update
                acc[i] += torch.randn_like(acc[i]) * (sigma * clip_c / m)
        w = [p + d for p, d in zip(w, acc)]
    set_w(model, w)
    return (*evaluate(model), clip_c)

rows = []
for frac in [1.0, 0.5, 0.2]:
    mase, prec, n_up, _ = fedavg(frac=frac)
    rows.append({"arm": f"participation {frac:.0%}", "MASE_median": mase,
                 "precision_up_pct": prec, "n_said_up": n_up})
    print(f"participation {frac:>4.0%}: MASE {mase:.4f}  prec_up {prec:5.1f}%  n_up {n_up}")

participation 100%: MASE 0.5072  prec_up  70.0%  n_up 383


participation  50%: MASE 0.5117  prec_up  69.5%  n_up 364


participation  20%: MASE 0.4895  prec_up  70.0%  n_up 443


## 2. Clipping and noise

Clip alone first (sigma 0), so the cost of bounding update norms is separated from the cost of the
noise itself. All privacy arms run at full participation, so the only change from the reference is
the mechanism being priced.

In [4]:
for sigma in [0.0, 0.5, 1.0]:
    mase, prec, n_up, clip_c = fedavg(frac=1.0, clip=True, sigma=sigma)   # the probe sets the bound
    label = f"clip @ median norm, sigma {sigma}"
    rows.append({"arm": label, "MASE_median": mase, "precision_up_pct": prec, "n_said_up": n_up})
    print(f"{label:28s}: MASE {mase:.4f}  prec_up {prec:5.1f}%  n_up {n_up}   (clip bound {clip_c:.3f})")

res = pd.DataFrame(rows).set_index("arm")
print()
print(res.round(3).to_string())
print()
print(f"read precision against the {base_rate:.1f}% base rate of {int(eligible.sum()):,} services")
print("sigma is a noise multiplier, not a privacy budget: a formal (epsilon, delta) claim needs an")
print("accountant over the whole run and is deliberately not made here.")
res.to_csv(CANON / "federation_stress.csv")
print("written: federation_stress.csv")

clip @ median norm, sigma 0.0: MASE 0.5076  prec_up  70.2%  n_up 389   (clip bound 0.713)


clip @ median norm, sigma 0.5: MASE 0.5039  prec_up  69.3%  n_up 319   (clip bound 0.713)


clip @ median norm, sigma 1.0: MASE 0.5100  prec_up  70.1%  n_up 294   (clip bound 0.713)

                               MASE_median  precision_up_pct  n_said_up
arm                                                                    
participation 100%                   0.507            69.974        383
participation 50%                    0.512            69.505        364
participation 20%                    0.490            69.977        443
clip @ median norm, sigma 0.0        0.508            70.180        389
clip @ median norm, sigma 0.5        0.504            69.279        319
clip @ median norm, sigma 1.0        0.510            70.068        294

read precision against the 45.9% base rate of 1,978 services
sigma is a noise multiplier, not a privacy budget: a formal (epsilon, delta) claim needs an
accountant over the whole run and is deliberately not made here.
written: federation_stress.csv


## What this settles

Whether the headline federation numbers survive realistic participation, and what clipping and
noise cost on the metric a board uses. One seed; the spread question belongs to `13`.